In [4]:
# -*- coding: utf-8 -*-
"""
Sticker cleaner v3 (robusto y estricto)
- Exime Hoja de Vida de Función Pública (HV-FP) -> no limpia.
- Detecta stickers con OCR ruidoso (digit-like mapping).
- Elimina SOLO las líneas del sticker (HV + número), con límites de seguridad.
- Genera reportes por archivo y globales (QA).

Ajusta:
  JSON_FOLDER = carpeta con tus .json
  OUTPUT_DIR  = carpeta de salidas
  COPY_IMAGES = True si quieres copiar PNG por categoría
"""

from __future__ import annotations
from pathlib import Path
import json, re, csv, shutil
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from collections import Counter, defaultdict

# ============================================================
# RUTAS / FLAGS
# ============================================================
JSON_FOLDER = r"D:\historias\dev\ocr_por_doc\LETRA A\ACTA N° 70\1" 
OUTPUT_DIR  = r"C:\Users\juans\Documents\version_final_historias laborales\answert"
COPY_IMAGES = False

DO_RUN = True
DEBUG = True
FORCE_SAVE_REPORTS_EVEN_IF_EMPTY = True

# ======================== CONFIGURACIÓN ==========================
@dataclass
class StickerConfig:
    # ... (lo que ya tienes)
    neighbor_radius: int = 1
    max_gap_lines: int = 6
    zone_fraction: float = 0.30
    block_size_limit: int = 5
    max_removed_ratio: float = 0.15

    min_digitlike_total: int = 12
    min_digitlike_ratio: float = 0.60
    max_tokens_for_sticker_line: int = 4
    min_digit_ratio_line: float = 0.85
    max_alpha_ratio_line: float = 0.20

    long_number_patterns: Tuple[re.Pattern, ...] = (
        re.compile(r"(?<!\d)(\d{11,})(?!\d)"),
        re.compile(r"\b\d{2,}(?:[:.\-_%]\d{2,}){2,}\d{2,}\b"),
        re.compile(r"(?:\d[\W_]?){12,}")
    )

    hv_regex: re.Pattern = re.compile(r"\bhoja\s*de\s*vida\b", re.I | re.U)
    hv_fp_regexes: Tuple[re.Pattern, ...] = (
        re.compile(r"\bformato\s+[úu]nico\s+de\s+hoja\s*de\s*vida\b", re.I),
        re.compile(r"\bhoja\s*de\s*vida.*funci[oó]n\s+p[úu]blica\b", re.I),
        re.compile(r"\bfunci[oó]n\s+p[úu]blica.*hoja\s*de\s*vida\b", re.I),
        re.compile(r"\bsigep\b", re.I),
        re.compile(r"\bdepartamento\s+administrativo\s+de\s+la\s+funci[oó]n\s+p[úu]blica\b", re.I),
    )
    hv_fp_extra_pattern: Optional[re.Pattern] = None

    # palabras negativas que invalidan números "sticker"
    negative_keywords: re.Pattern = re.compile(
        r"\b(telefono|tel\.?|línea|linea|nit\.?|nit|www|http|https|p[aá]gina|pagina|eps|pbx|celular|correo|web|"
        r"identificaci[oó]n|n[úu]mero|no\.|decreto|art(?:[íi]culo|\.)?|registro|fecha|certificado|"
        r"calle|cra\.?|carrera|av\.?|avenida|dir(?:ecci[oó]n)?|nº|n°|no\.?)\b", re.I
    )


    # ✅ NUEVO: criterios para que "HOJA DE VIDA" cuente como sticker real
    hv_sticker_max_tokens: int = 4          # encabezado corto
    hv_sticker_max_chars: int  = 40         # y con pocos caracteres
    hv_sticker_edge_bias: bool = True       # si está en 30% sup/inf, se flexibiliza


CFG = StickerConfig()

# ========================== UTILIDADES ===========================
def is_hv_sticker_line(line: str, idx: int, n_lines: int, cfg: StickerConfig = CFG) -> bool:
    """Solo considera 'HOJA DE VIDA' como sticker si está aislado/corto o en borde."""
    if not is_hv_robust(line, cfg):
        return False
    if is_content_rich_line(line):
        return False
    # corto/aislado
    if token_count(line) <= cfg.hv_sticker_max_tokens and len(line) <= cfg.hv_sticker_max_chars:
        return True
    # si está en bordes, permitimos un poco más largo
    if cfg.hv_sticker_edge_bias and in_top_or_bottom_zone(idx, n_lines, cfg.zone_fraction):
        if token_count(line) <= (cfg.hv_sticker_max_tokens + 1) and len(line) <= (cfg.hv_sticker_max_chars + 10):
            return True
    return False


def normalize_ocr_text(t: Optional[str]) -> str:
    if not t: return ""
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = re.sub(r"[·•◦▪●■□▫▶►♦●]", " ", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = "\n".join(ln.strip() for ln in t.split("\n"))
    return t

def split_lines(t: str) -> List[str]: return t.split("\n") if t else []

def token_count(s: str) -> int: return len([x for x in s.strip().split(" ") if x])

def char_ratio(s: str) -> Tuple[float, float, float]:
    n = max(len(s), 1)
    return (sum(c.isdigit() for c in s) / n,
            sum(c.isalpha() for c in s) / n,
            s.count("-") / n)

def collapse_letters(s: str) -> str: return re.sub(r"[^A-ZÁÉÍÓÚÜÑ]", "", s.upper())

def is_hv_robust(line: str, cfg: StickerConfig = CFG) -> bool:
    return bool(cfg.hv_regex.search(line) or ("HOJADEVIDA" in collapse_letters(line)))

def has_long_number_patterns(s: str, cfg: StickerConfig = CFG) -> bool:
    return any(p.search(s) for p in cfg.long_number_patterns)

# digit-like mapping
_DIGITLIKE_MAP = {"O":"0","o":"0","Q":"0","D":"0","I":"1","l":"1","L":"1","i":"1","|":"1","!":"1",
                  "Z":"2","z":"2","S":"5","s":"5","$":"5","B":"8","b":"8","G":"6","g":"6","T":"7","t":"7"}

def to_digitlike(s: str) -> str:
    out = []
    for ch in s:
        if ch.isdigit() or ch in ":.-_%": out.append(ch)
        elif ch in _DIGITLIKE_MAP: out.append(_DIGITLIKE_MAP[ch])
    return "".join(out)

def digitlike_stats(s: str) -> Tuple[int, float]:
    mapped = to_digitlike(s)
    cnt = sum(c.isdigit() for c in mapped)
    ratio = cnt / max(len(s), 1)
    return cnt, ratio

# contenido rico (párrafos reales)
_CONTENT_WORDS = re.compile(
    r"\b(nombrad[oa]|cargo|oficio|resolu(?:ci[oó]n)?|fecha|ciudad|direcci[oó]n|tel[eé]fono|departamento|"
    r"certificad[oa]|aprobaci[oó]n|observaci[oó]n|concepto|historia|ingreso|ministerio|servicio|m[eé]dico|"
    r"padre|madre|c[oó]nyuge|barrio|correo|profesi[oó]n|estudios|empresa)\b", re.I
)
def is_content_rich_line(line: str) -> bool:
    d_ratio, a_ratio, _ = char_ratio(line)
    return (token_count(line) >= 6 and a_ratio >= 0.55) or bool(_CONTENT_WORDS.search(line))

def in_top_or_bottom_zone(idx: int, n: int, frac: float) -> bool:
    zone = max(1, int(frac * n)); return idx < zone or idx >= (n - zone)

# ========= NÚMERO STICKER (línea completa) vs. CÓDIGO EMBEBIDO =========
def is_sticker_like_number_line(line: str, cfg: StickerConfig = CFG) -> bool:
    """Para ELIMINAR línea completa: corta, poco alfabeto, sin palabras negativas."""
    if cfg.negative_keywords.search(line):
        return False
    if is_content_rich_line(line):
        return False
    if has_long_number_patterns(line, cfg):
        return True
    cnt, ratio = digitlike_stats(line)
    d_ratio, a_ratio, _ = char_ratio(line)
    if token_count(line) <= cfg.max_tokens_for_sticker_line and (
        (cnt >= cfg.min_digitlike_total and ratio >= cfg.min_digitlike_ratio) or
        (d_ratio >= cfg.min_digit_ratio_line and a_ratio <= cfg.max_alpha_ratio_line)
    ):
        return True
    return False


# patrón para CÓDIGO EMBEBIDO (dentro de una frase) – clase reducida de letras confusas
_EMBED_CHARS = r"0-9OQDGILZSBGT"
EMBED_BARCODE_RE = re.compile(rf"([{_EMBED_CHARS}]{{2,}}(?:[:.\-_%][{_EMBED_CHARS}]{{2,}}){{2,}})")

def find_inline_barcode_spans(line: str) -> List[Tuple[int,int,str]]:
    """
    Devuelve spans (start, end, matched_text) de posibles códigos embebidos.
    Valida con digitlike_stats y exige al menos 2 separadores.
    """
    spans = []
    for m in EMBED_BARCODE_RE.finditer(line):
        s, e = m.span()
        seg = m.group(0)
        # separadores presentes
        seps = sum(seg.count(ch) for ch in ":.-_%")
        cnt, _ = digitlike_stats(seg)
        if seps >= 2 and cnt >= 12:
            spans.append((s, e, seg))
    return spans

# ====================== DETECCIÓN Y LIMPIEZA ======================
def find_sticker_block(lines: List[str], cfg: StickerConfig = CFG) -> Tuple[bool, List[int], bool]:
    """
    Elimina SOLO:
      - la línea de HV si es encabezado corto/aislado (no párrafo),
      - la línea de número tipo sticker (corta).
    Si HV está metido en un párrafo, NO se elimina (se tratará solo el número).
    También permite “solo número” si el número está en línea tipo-sticker.
    """
    n = len(lines)
    if n == 0:
        return False, [], False

    hv_idxs = [i for i, ln in enumerate(lines) if is_hv_sticker_line(ln, i, n, cfg)]
    num_idxs = [i for i, ln in enumerate(lines) if is_sticker_like_number_line(ln, cfg)]

    # caso sin número claro → no hacemos nada (no borrar párrafos)
    if not num_idxs:
        return False, [], False

    # fallback: si hay HV sticker, emparejamos; si NO hay HV sticker, aceptamos sólo número si está en bordes o muy aislado
    if hv_idxs:
        # empareja por mínima distancia
        best_pair, best_dist = None, 10**9
        for h in hv_idxs:
            for m in num_idxs:
                d = abs(h - m)
                if d < best_dist:
                    best_dist, best_pair = d, (h, m)
        h, m = best_pair
        close = best_dist <= cfg.max_gap_lines
        edges = in_top_or_bottom_zone(h, n, cfg.zone_fraction) and in_top_or_bottom_zone(m, n, cfg.zone_fraction)
        if not (close or edges):
            # no suficientemente relacionados → sólo evaluamos “sólo número”
            hv_idxs = []
        else:
            block = {h, m}
    if not hv_idxs:
        # sólo número: exigimos que esté en borde o muy aislado
        # (esto evita confundir teléfonos/NIT en medio de párrafos)
        edge_nums = [i for i in num_idxs if in_top_or_bottom_zone(i, n, cfg.zone_fraction)]
        if not edge_nums:
            return False, [], False
        m = edge_nums[0]
        block = {m}

    # expande ±1 SOLO si también es línea tipo-sticker y no es contenido rico
    for idx in list(block):
        for j in (idx - cfg.neighbor_radius, idx + cfg.neighbor_radius):
            if 0 <= j < n:
                ln = lines[j]
                if not is_content_rich_line(ln) and (is_hv_sticker_line(ln, j, n, cfg) or is_sticker_like_number_line(ln, cfg)):
                    block.add(j)

    # límites de seguridad
    if len(block) > cfg.block_size_limit or (len(block) / max(1, n)) > cfg.max_removed_ratio:
        # nos quedamos solo con el número (o con HV+num si eran 2)
        if len(block) >= 2:
            # prioriza número
            num_only = [i for i in sorted(block) if is_sticker_like_number_line(lines[i], cfg)]
            block = set(num_only[:1])
        else:
            # ya era 1 línea
            pass

    removed = sorted(block)
    hv_in_sticker = any(is_hv_sticker_line(lines[i], i, n, cfg) for i in removed)
    return True, removed, hv_in_sticker


def redact_inline_codes(lines: List[str]) -> Tuple[List[str], List[str]]:
    """
    Reemplaza códigos embebidos por [BARCODE] en líneas 'content-rich'.
    No elimina líneas completas.
    Devuelve (nuevas líneas, lista de recortes sustituidos para el preview).
    """
    redacted_lines = []
    previews = []
    for ln in lines:
        spans = find_inline_barcode_spans(ln)
        if spans and is_content_rich_line(ln):
            # reemplaza de derecha a izquierda para no mover offsets
            s_ln = ln
            for s, e, seg in sorted(spans, key=lambda x: x[0], reverse=True):
                s_ln = s_ln[:s] + "[BARCODE]" + s_ln[e:]
                previews.append(seg)
            redacted_lines.append(s_ln)
        else:
            redacted_lines.append(ln)
    return redacted_lines, previews

# ================== HV-FP (exención) + QA ===================
def is_hv_funcion_publica(text: str, cfg: StickerConfig = CFG) -> bool:
    if any(p.search(text) for p in cfg.hv_fp_regexes): return True
    if cfg.hv_fp_extra_pattern and cfg.hv_fp_extra_pattern.search(text): return True
    return False

def hv_outside_sticker(lines: List[str], idxs_removed: List[int]) -> bool:
    removed = set(idxs_removed)
    for i, ln in enumerate(lines):
        if i in removed: continue
        if is_hv_robust(ln): return True
    return False

def extract_removed_text(lines: List[str], idxs_removed: List[int], window:int=1) -> str:
    n = len(lines); parts = []
    for i in idxs_removed:
        i0 = max(0, i - window); i1 = min(n, i + window + 1)
        parts.append(f"[{i}] " + " | ".join(lines[i0:i1]))
    return " || ".join(parts)


QA_CATEGORIES = (
    "HV_FP_SKIP",
    "STICKER_ONLY",
    "BARCODE_ONLY",
    "STICKER_AND_HV_UNCERTAIN",
    "HV_REAL_TEXTUAL",
    "LONG_NUMBER_ONLY",
    "NO_STICKER"
)

def categorize_page_after(res: Dict) -> str:
    if res.get("hv_funcion_publica_skip"): return "HV_FP_SKIP"
    b = res["barcode_flag"]; hv_in = res["hv_in_sticker"]; hv_out = res["hv_outside_sticker"]
    raw = res["ocr_text_raw"] or ""; clean = res["ocr_text_clean"] or ""
    if b:
        if hv_in and not hv_out: return "STICKER_ONLY"
        if hv_in and hv_out:     return "STICKER_AND_HV_UNCERTAIN"
        return "BARCODE_ONLY"
    else:
        # si quedan patrones largos, marcamos LONG_NUMBER_ONLY
        if any(p.search(raw) for p in CFG.long_number_patterns) or any(p.search(clean) for p in CFG.long_number_patterns):
            return "LONG_NUMBER_ONLY"
        if is_hv_robust(clean):  # HV textual fuera de sticker
            return "HV_REAL_TEXTUAL"
        return "NO_STICKER"

# ======================== I/O + PROCESO ========================
def safe_read_json(fp: Path):
    txt = fp.read_text(encoding="utf-8", errors="replace")
    data = json.loads(txt)
    if isinstance(data, dict) and "paginas" in data: data = data["paginas"]
    if not isinstance(data, list): raise ValueError("JSON debe ser lista de páginas o dict{'paginas':[...]}.")
    return data

def load_histories(folder: Path) -> List[Tuple[Path, List[Dict]]]:
    out = []
    for fp in sorted(folder.glob("*.json")):
        if fp.name.endswith("_clean.json") or fp.name.endswith("_clean_report.json"): continue
        try:
            data = safe_read_json(fp); out.append((fp, data))
        except Exception as e:
            print(f"[WARN] No pude leer {fp.name}: {e}")
    return out

def debug_dump_lines(file_name: str, lines: List[str], rm_idxs: List[int], folder: Path):
    out_csv = folder / f"{Path(file_name).stem}_debug_sticker.csv"
    with out_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["idx","removed","digitlike_cnt","digitlike_ratio","digit_ratio","alpha_ratio","long_number","has_HV","line"])
        rm = set(rm_idxs)
        for i, ln in enumerate(lines):
            dl_cnt, dl_ratio = digitlike_stats(ln)
            d_ratio, a_ratio, _ = char_ratio(ln)
            w.writerow([i, int(i in rm), dl_cnt, round(dl_ratio,3), round(d_ratio,3), round(a_ratio,3),
                        int(has_long_number_patterns(ln)), int(is_hv_robust(ln)), ln])

def process_history_json(pages: List[Dict], dbg_path: Optional[Path]=None) -> List[Dict]:
    results = []
    for row in pages:
        texto_raw = row.get("texto", "") or ""
        norm = normalize_ocr_text(texto_raw)
        lines = split_lines(norm)

        # 1) Exención HV-FP
        if is_hv_funcion_publica(norm):
            res = {
                "pagina": row.get("pagina"), "imagen": row.get("imagen"),
                "hv_funcion_publica_skip": True, "barcode_flag": False,
                "removed_line_idxs": [], "removed_text_preview": "",
                "inline_redactions": [], "hv_in_sticker": False, "hv_outside_sticker": False,
                "ocr_text_raw": texto_raw, "ocr_text_clean": texto_raw
            }
            res["qa_category"] = "HV_FP_SKIP"
            results.append(res); continue

        # 2) Detectar bloque (líneas cortas) y limpiar
        barcode_flag, rm_idxs, hv_in = find_sticker_block(lines, cfg=CFG)
        kept_lines = [ln for i, ln in enumerate(lines) if i not in set(rm_idxs)]

        # 3) Redactar códigos embebidos (en párrafos)
        redacted_lines, previews_inline = redact_inline_codes(kept_lines)
        clean_text = "\n".join(redacted_lines)
        hv_out = hv_outside_sticker(lines, rm_idxs)

        removed_preview = extract_removed_text(lines, rm_idxs, window=1)

        if DEBUG and dbg_path is not None:
            try: debug_dump_lines(dbg_path.name, lines, rm_idxs, dbg_path.parent)
            except Exception as e: print(f"[WARN] Debug no escrito para {dbg_path.name}: {e}")

        res = {
            "pagina": row.get("pagina"), "imagen": row.get("imagen"),
            "hv_funcion_publica_skip": False, "barcode_flag": barcode_flag,
            "removed_line_idxs": rm_idxs, "removed_text_preview": removed_preview,
            "inline_redactions": previews_inline,  # <<-- NUEVO: lista de substrings sustituidos
            "hv_in_sticker": hv_in, "hv_outside_sticker": hv_out,
            "ocr_text_raw": texto_raw, "ocr_text_clean": clean_text
        }
        res["qa_category"] = categorize_page_after(res)
        results.append(res)
    return results

def ensure_dir(p: Path): p.mkdir(parents=True, exist_ok=True)

def copy_image_safe(src: Optional[str], dst_dir: Path):
    if not src: return
    try:
        src_p = Path(src)
        if src_p.exists():
            ensure_dir(dst_dir); shutil.copy2(src_p, dst_dir / src_p.name)
    except Exception as e:
        print(f"[WARN] No se pudo copiar imagen {src}: {e}")

def process_folder(folder_path: str, output_dir: str, save_outputs: bool = True, copy_images: bool = False) -> List[Dict]:
    folder = Path(folder_path); outdir = Path(output_dir); ensure_dir(outdir)
    histories = load_histories(folder)
    if not histories:
        print("[INFO] No se encontraron JSON."); return []

    all_rows: List[Dict] = []; per_file_counts = defaultdict(Counter)
    for fp, data in histories:
        processed = process_history_json(data, dbg_path=fp)

        # por archivo
        if save_outputs or FORCE_SAVE_REPORTS_EVEN_IF_EMPTY:
            out_json = fp.with_name(fp.stem + "_clean.json")
            out_json.write_text(json.dumps(processed, ensure_ascii=False, indent=2), encoding="utf-8")
            out_csv = fp.with_name(fp.stem + "_clean_report.csv")
            with out_csv.open("w", newline="", encoding="utf-8") as f:
                w = csv.writer(f)
                w.writerow(["file","pagina","qa_category","barcode_flag","hv_in_sticker","hv_outside_sticker",
                            "hv_funcion_publica_skip","removed_line_idxs","inline_redactions"])
                for r in processed:
                    w.writerow([fp.name, r["pagina"], r["qa_category"], r["barcode_flag"], r["hv_in_sticker"],
                                r["hv_outside_sticker"], r["hv_funcion_publica_skip"],
                                "|".join(map(str, r["removed_line_idxs"])), " || ".join(r.get("inline_redactions",[]))])

        for r in processed:
            rr = {**r, "file": fp.name}; all_rows.append(rr)
            per_file_counts[fp.name][r["qa_category"]] += 1
            if copy_images and r.get("imagen"):
                cat_dir = outdir / "examples" / r["qa_category"]; copy_image_safe(r["imagen"], cat_dir)

    # global examples
    examples_csv = Path(output_dir) / "sticker_QA_examples.csv"
    with examples_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["file","pagina","qa_category","barcode_flag","hv_in_sticker","hv_outside_sticker",
                    "hv_funcion_publica_skip","image","removed_text_preview","inline_redactions"])
        for r in all_rows:
            w.writerow([r["file"], r["pagina"], r["qa_category"], r["barcode_flag"], r["hv_in_sticker"],
                        r["hv_outside_sticker"], r["hv_funcion_publica_skip"], r.get("imagen",""),
                        r.get("removed_text_preview",""), " || ".join(r.get("inline_redactions",[]))])

    summary_csv = Path(output_dir) / "sticker_QA_summary.csv"
    with summary_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); header = ["file"] + list(QA_CATEGORIES); w.writerow(header)
        for fname, cnt in per_file_counts.items():
            row = [fname] + [cnt.get(cat, 0) for cat in QA_CATEGORIES]; w.writerow(row)

    total = len(all_rows); flagged = sum(1 for r in all_rows if r["barcode_flag"])
    skipped = sum(1 for r in all_rows if r.get("hv_funcion_publica_skip"))
    print(f"[RESUMEN] archivos: {len(histories)} | páginas: {total} | con_sticker: {flagged} | HV-FP exentas: {skipped}")
    return all_rows

# ========================== EJECUCIÓN ==========================
if DO_RUN:
    _ = process_folder(JSON_FOLDER, OUTPUT_DIR, save_outputs=True, copy_images=COPY_IMAGES)

[RESUMEN] archivos: 12 | páginas: 500 | con_sticker: 368 | HV-FP exentas: 31


In [ ]:
# -*- coding: utf-8 -*-
"""
Sticker cleaner v4 (estricto + robusto + redacción inline)
- Exime HV Función Pública.
- Detecta códigos con OCR ruidoso (digit-like mapping).
- Evita falsos positivos (teléfonos, NIT, URLs, etc.).
- Si el código está en una línea corta tipo sticker → elimina la línea.
- Si el código está embebido en un párrafo → redáctalo (sustitución parcial) y conserva la línea.
- Reporta qué se eliminó y qué se redactó.
"""

from __future__ import annotations
from pathlib import Path
import json, re, csv, shutil
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from collections import Counter, defaultdict

# ========================= RUTAS / FLAGS =========================
JSON_FOLDER = r"D:\historias\dev\ocr_por_doc\LETRA A\ACTA N° 70\1" 
OUTPUT_DIR  = r"C:\Users\juans\Documents\version_final_historias laborales\answert_v2"
COPY_IMAGES = False

DO_RUN = True
DEBUG = True
FORCE_SAVE_REPORTS_EVEN_IF_EMPTY = True

# ======================== CONFIGURACIÓN ==========================
@dataclass
class StickerConfig:
    # límites estrictos al eliminar líneas completas
    neighbor_radius: int = 1
    max_gap_lines: int = 6
    zone_fraction: float = 0.30
    block_size_limit: int = 5
    max_removed_ratio: float = 0.15

    # detección de código con OCR ruidoso
    min_digitlike_total: int = 12
    min_digitlike_ratio: float = 0.60
    max_tokens_for_sticker_line: int = 4
    min_digit_ratio_line: float = 0.85
    max_alpha_ratio_line: float = 0.20

    # patrón general con separadores (apoya el matching)
    long_number_patterns: Tuple[re.Pattern, ...] = (
        re.compile(r"(?<!\d)(\d{11,})(?!\d)"),
        re.compile(r"\b\d{2,}(?:[:.\-_%]\d{2,}){2,}\d{2,}\b"),
        re.compile(r"(?:\d[\W_]?){12,}")
    )

    # “Hoja de Vida” (robusto)
    hv_regex: re.Pattern = re.compile(r"\bhoja\s*de\s*vida\b", re.I | re.U)

    # HV Función Pública (exención)
    hv_fp_regexes: Tuple[re.Pattern, ...] = (
        re.compile(r"\bformato\s+[úu]nico\s+de\s+hoja\s*de\s*vida\b", re.I),
        re.compile(r"\bhoja\s*de\s*vida.*funci[oó]n\s+p[úu]blica\b", re.I),
        re.compile(r"\bfunci[oó]n\s+p[úu]blica.*hoja\s*de\s*vida\b", re.I),
        re.compile(r"\bsigep\b", re.I),
        re.compile(r"\bdepartamento\s+administrativo\s+de\s+la\s+funci[oó]n\s+p[úu]blica\b", re.I),
    )
    hv_fp_extra_pattern: Optional[re.Pattern] = None   # pega tu regex extra aquí si quieres

    # NEGATIVOS: si aparecen en la misma línea del número → NO es sticker
    negative_keywords: re.Pattern = re.compile(
        r"\b(telefono|tel\.?|línea|linea|nit\.?|nit|www|http|p[aá]gina|pagina|eps|pbx|celular|correo|web|"
        r"identificaci[oó]n|n[úu]mero|no\.)\b", re.I
    )

CFG = StickerConfig()

# ========================== UTILIDADES ===========================
def normalize_ocr_text(t: Optional[str]) -> str:
    if not t: return ""
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = re.sub(r"[·•◦▪●■□▫▶►♦●]", " ", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = "\n".join(ln.strip() for ln in t.split("\n"))
    return t

def split_lines(t: str) -> List[str]: return t.split("\n") if t else []

def token_count(s: str) -> int: return len([x for x in s.strip().split(" ") if x])

def char_ratio(s: str) -> Tuple[float, float, float]:
    n = max(len(s), 1)
    return (sum(c.isdigit() for c in s) / n,
            sum(c.isalpha() for c in s) / n,
            s.count("-") / n)

def collapse_letters(s: str) -> str: return re.sub(r"[^A-ZÁÉÍÓÚÜÑ]", "", s.upper())

def is_hv_robust(line: str, cfg: StickerConfig = CFG) -> bool:
    return bool(cfg.hv_regex.search(line) or ("HOJADEVIDA" in collapse_letters(line)))

def has_long_number_patterns(s: str, cfg: StickerConfig = CFG) -> bool:
    return any(p.search(s) for p in cfg.long_number_patterns)

# digit-like mapping
_DIGITLIKE_MAP = {"O":"0","o":"0","Q":"0","D":"0","I":"1","l":"1","L":"1","i":"1","|":"1","!":"1",
                  "Z":"2","z":"2","S":"5","s":"5","$":"5","B":"8","b":"8","G":"6","g":"6","T":"7","t":"7"}

def to_digitlike(s: str) -> str:
    out = []
    for ch in s:
        if ch.isdigit() or ch in ":.-_%": out.append(ch)
        elif ch in _DIGITLIKE_MAP: out.append(_DIGITLIKE_MAP[ch])
    return "".join(out)

def digitlike_stats(s: str) -> Tuple[int, float]:
    mapped = to_digitlike(s)
    cnt = sum(c.isdigit() for c in mapped)
    ratio = cnt / max(len(s), 1)
    return cnt, ratio

# contenido rico (párrafos reales)
_CONTENT_WORDS = re.compile(
    r"\b(nombrad[oa]|cargo|oficio|resolu(?:ci[oó]n)?|fecha|ciudad|direcci[oó]n|tel[eé]fono|departamento|"
    r"certificad[oa]|aprobaci[oó]n|observaci[oó]n|concepto|historia|ingreso|ministerio|servicio|m[eé]dico|"
    r"padre|madre|c[oó]nyuge|barrio|correo|profesi[oó]n|estudios|empresa)\b", re.I
)
def is_content_rich_line(line: str) -> bool:
    d_ratio, a_ratio, _ = char_ratio(line)
    return (token_count(line) >= 6 and a_ratio >= 0.55) or bool(_CONTENT_WORDS.search(line))

def in_top_or_bottom_zone(idx: int, n: int, frac: float) -> bool:
    zone = max(1, int(frac * n)); return idx < zone or idx >= (n - zone)

# ========= NÚMERO STICKER (línea completa) vs. CÓDIGO EMBEBIDO =========
def is_sticker_like_number_line(line: str, cfg: StickerConfig = CFG) -> bool:
    """Solo para ELIMINAR la línea completa: corta, poco alfabeto, sin palabras negativas."""
    if cfg.negative_keywords.search(line): return False
    if is_content_rich_line(line): return False
    if has_long_number_patterns(line, cfg): return True
    cnt, ratio = digitlike_stats(line)
    d_ratio, a_ratio, _ = char_ratio(line)
    if token_count(line) <= cfg.max_tokens_for_sticker_line and (
        (cnt >= cfg.min_digitlike_total and ratio >= cfg.min_digitlike_ratio) or
        (d_ratio >= cfg.min_digit_ratio_line and a_ratio <= cfg.max_alpha_ratio_line)
    ):
        return True
    return False

# patrón para CÓDIGO EMBEBIDO (dentro de una frase) – clase reducida de letras confusas
_EMBED_CHARS = r"0-9OQDGILZSBGT"
EMBED_BARCODE_RE = re.compile(rf"([{_EMBED_CHARS}]{{2,}}(?:[:.\-_%][{_EMBED_CHARS}]{{2,}}){{2,}})")

def find_inline_barcode_spans(line: str) -> List[Tuple[int,int,str]]:
    """
    Devuelve spans (start, end, matched_text) de posibles códigos embebidos.
    Valida con digitlike_stats y exige al menos 2 separadores.
    """
    spans = []
    for m in EMBED_BARCODE_RE.finditer(line):
        s, e = m.span()
        seg = m.group(0)
        # separadores presentes
        seps = sum(seg.count(ch) for ch in ":.-_%")
        cnt, _ = digitlike_stats(seg)
        if seps >= 2 and cnt >= 12:
            spans.append((s, e, seg))
    return spans

# ====================== DETECCIÓN Y LIMPIEZA ======================
def find_sticker_block(lines: List[str], cfg: StickerConfig = CFG) -> Tuple[bool, List[int], bool]:
    """
    Elimina SOLO la línea 'HOJA DE VIDA' y la línea con número tipo sticker (corta).
    Si están muy cerca (<= max_gap_lines) o ambos en zona de borde, se considera bloque.
    """
    n = len(lines)
    if n == 0: return False, [], False

    hv_idxs = [i for i, ln in enumerate(lines) if is_hv_robust(ln, cfg)]
    num_idxs = [i for i, ln in enumerate(lines) if is_sticker_like_number_line(ln, cfg)]

    if not hv_idxs or not num_idxs: return False, [], False

    # pareo por distancia mínima
    best = None; best_dist = 10**9
    for h in hv_idxs:
        for m in num_idxs:
            d = abs(h - m)
            if d < best_dist: best_dist, best = d, (h, m)

    h, m = best
    close = best_dist <= cfg.max_gap_lines
    edges = in_top_or_bottom_zone(h, n, cfg.zone_fraction) and in_top_or_bottom_zone(m, n, cfg.zone_fraction)
    if not (close or edges): return False, [], False

    block = {h, m}
    # vecinas inmediatas si también parecen sticker y no son contenido rico
    for idx in list(block):
        for j in (idx - cfg.neighbor_radius, idx + cfg.neighbor_radius):
            if 0 <= j < n:
                ln = lines[j]
                if not is_content_rich_line(ln) and (is_hv_robust(ln, cfg) or is_sticker_like_number_line(ln, cfg)):
                    block.add(j)

    # límites de seguridad
    if len(block) > cfg.block_size_limit or (len(block) / max(1, n)) > cfg.max_removed_ratio:
        block = {h, m}

    return True, sorted(block), (h in block)

def redact_inline_codes(lines: List[str]) -> Tuple[List[str], List[str]]:
    """
    Reemplaza códigos embebidos por [BARCODE] en líneas 'content-rich'.
    No elimina líneas completas.
    Devuelve (nuevas líneas, lista de recortes sustituidos para el preview).
    """
    redacted_lines = []
    previews = []
    for ln in lines:
        spans = find_inline_barcode_spans(ln)
        if spans and is_content_rich_line(ln):
            # reemplaza de derecha a izquierda para no mover offsets
            s_ln = ln
            for s, e, seg in sorted(spans, key=lambda x: x[0], reverse=True):
                s_ln = s_ln[:s] + "[BARCODE]" + s_ln[e:]
                previews.append(seg)
            redacted_lines.append(s_ln)
        else:
            redacted_lines.append(ln)
    return redacted_lines, previews

# ================== HV-FP (exención) + QA ===================
def is_hv_funcion_publica(text: str, cfg: StickerConfig = CFG) -> bool:
    if any(p.search(text) for p in cfg.hv_fp_regexes): return True
    if cfg.hv_fp_extra_pattern and cfg.hv_fp_extra_pattern.search(text): return True
    return False

def hv_outside_sticker(lines: List[str], idxs_removed: List[int]) -> bool:
    removed = set(idxs_removed)
    for i, ln in enumerate(lines):
        if i in removed: continue
        if is_hv_robust(ln): return True
    return False

def extract_removed_text(lines: List[str], idxs_removed: List[int], window:int=1) -> str:
    n = len(lines); parts = []
    for i in idxs_removed:
        i0 = max(0, i - window); i1 = min(n, i + window + 1)
        parts.append(f"[{i}] " + " | ".join(lines[i0:i1]))
    return " || ".join(parts)

QA_CATEGORIES = (
    "HV_FP_SKIP",
    "STICKER_ONLY",
    "BARCODE_ONLY",
    "STICKER_AND_HV_UNCERTAIN",
    "HV_REAL_TEXTUAL",
    "LONG_NUMBER_ONLY",
    "NO_STICKER"
)

def categorize_page_after(res: Dict) -> str:
    if res.get("hv_funcion_publica_skip"): return "HV_FP_SKIP"
    b = res["barcode_flag"]; hv_in = res["hv_in_sticker"]; hv_out = res["hv_outside_sticker"]
    raw = res["ocr_text_raw"] or ""; clean = res["ocr_text_clean"] or ""
    if b:
        if hv_in and not hv_out: return "STICKER_ONLY"
        if hv_in and hv_out:     return "STICKER_AND_HV_UNCERTAIN"
        return "BARCODE_ONLY"
    else:
        # si quedan patrones largos, marcamos LONG_NUMBER_ONLY
        if any(p.search(raw) for p in CFG.long_number_patterns) or any(p.search(clean) for p in CFG.long_number_patterns):
            return "LONG_NUMBER_ONLY"
        if is_hv_robust(clean):  # HV textual fuera de sticker
            return "HV_REAL_TEXTUAL"
        return "NO_STICKER"

# ======================== I/O + PROCESO ========================
def safe_read_json(fp: Path):
    txt = fp.read_text(encoding="utf-8", errors="replace")
    data = json.loads(txt)
    if isinstance(data, dict) and "paginas" in data: data = data["paginas"]
    if not isinstance(data, list): raise ValueError("JSON debe ser lista de páginas o dict{'paginas':[...]}.")
    return data

def load_histories(folder: Path) -> List[Tuple[Path, List[Dict]]]:
    out = []
    for fp in sorted(folder.glob("*.json")):
        if fp.name.endswith("_clean.json") or fp.name.endswith("_clean_report.json"): continue
        try:
            data = safe_read_json(fp); out.append((fp, data))
        except Exception as e:
            print(f"[WARN] No pude leer {fp.name}: {e}")
    return out

def debug_dump_lines(file_name: str, lines: List[str], rm_idxs: List[int], folder: Path):
    out_csv = folder / f"{Path(file_name).stem}_debug_sticker.csv"
    with out_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["idx","removed","digitlike_cnt","digitlike_ratio","digit_ratio","alpha_ratio","long_number","has_HV","line"])
        rm = set(rm_idxs)
        for i, ln in enumerate(lines):
            dl_cnt, dl_ratio = digitlike_stats(ln)
            d_ratio, a_ratio, _ = char_ratio(ln)
            w.writerow([i, int(i in rm), dl_cnt, round(dl_ratio,3), round(d_ratio,3), round(a_ratio,3),
                        int(has_long_number_patterns(ln)), int(is_hv_robust(ln)), ln])

def process_history_json(pages: List[Dict], dbg_path: Optional[Path]=None) -> List[Dict]:
    results = []
    for row in pages:
        texto_raw = row.get("texto", "") or ""
        norm = normalize_ocr_text(texto_raw)
        lines = split_lines(norm)

        # 1) Exención HV-FP
        if is_hv_funcion_publica(norm):
            res = {
                "pagina": row.get("pagina"), "imagen": row.get("imagen"),
                "hv_funcion_publica_skip": True, "barcode_flag": False,
                "removed_line_idxs": [], "removed_text_preview": "",
                "inline_redactions": [], "hv_in_sticker": False, "hv_outside_sticker": False,
                "ocr_text_raw": texto_raw, "ocr_text_clean": texto_raw
            }
            res["qa_category"] = "HV_FP_SKIP"
            results.append(res); continue

        # 2) Detectar bloque (líneas cortas) y limpiar
        barcode_flag, rm_idxs, hv_in = find_sticker_block(lines, cfg=CFG)
        kept_lines = [ln for i, ln in enumerate(lines) if i not in set(rm_idxs)]

        # 3) Redactar códigos embebidos (en párrafos)
        redacted_lines, previews_inline = redact_inline_codes(kept_lines)
        clean_text = "\n".join(redacted_lines)
        hv_out = hv_outside_sticker(lines, rm_idxs)

        removed_preview = extract_removed_text(lines, rm_idxs, window=1)

        if DEBUG and dbg_path is not None:
            try: debug_dump_lines(dbg_path.name, lines, rm_idxs, dbg_path.parent)
            except Exception as e: print(f"[WARN] Debug no escrito para {dbg_path.name}: {e}")

        res = {
            "pagina": row.get("pagina"), "imagen": row.get("imagen"),
            "hv_funcion_publica_skip": False, "barcode_flag": barcode_flag,
            "removed_line_idxs": rm_idxs, "removed_text_preview": removed_preview,
            "inline_redactions": previews_inline,  # <<-- NUEVO: lista de substrings sustituidos
            "hv_in_sticker": hv_in, "hv_outside_sticker": hv_out,
            "ocr_text_raw": texto_raw, "ocr_text_clean": clean_text
        }
        res["qa_category"] = categorize_page_after(res)
        results.append(res)
    return results

def ensure_dir(p: Path): p.mkdir(parents=True, exist_ok=True)

def copy_image_safe(src: Optional[str], dst_dir: Path):
    if not src: return
    try:
        src_p = Path(src)
        if src_p.exists():
            ensure_dir(dst_dir); shutil.copy2(src_p, dst_dir / src_p.name)
    except Exception as e:
        print(f"[WARN] No se pudo copiar imagen {src}: {e}")

def process_folder(folder_path: str, output_dir: str, save_outputs: bool = True, copy_images: bool = False) -> List[Dict]:
    folder = Path(folder_path); outdir = Path(output_dir); ensure_dir(outdir)
    histories = load_histories(folder)
    if not histories:
        print("[INFO] No se encontraron JSON."); return []

    all_rows: List[Dict] = []; per_file_counts = defaultdict(Counter)
    for fp, data in histories:
        processed = process_history_json(data, dbg_path=fp)

        # por archivo
        if save_outputs or FORCE_SAVE_REPORTS_EVEN_IF_EMPTY:
            out_json = fp.with_name(fp.stem + "_clean.json")
            out_json.write_text(json.dumps(processed, ensure_ascii=False, indent=2), encoding="utf-8")
            out_csv = fp.with_name(fp.stem + "_clean_report.csv")
            with out_csv.open("w", newline="", encoding="utf-8") as f:
                w = csv.writer(f)
                w.writerow(["file","pagina","qa_category","barcode_flag","hv_in_sticker","hv_outside_sticker",
                            "hv_funcion_publica_skip","removed_line_idxs","inline_redactions"])
                for r in processed:
                    w.writerow([fp.name, r["pagina"], r["qa_category"], r["barcode_flag"], r["hv_in_sticker"],
                                r["hv_outside_sticker"], r["hv_funcion_publica_skip"],
                                "|".join(map(str, r["removed_line_idxs"])), " || ".join(r.get("inline_redactions",[]))])

        for r in processed:
            rr = {**r, "file": fp.name}; all_rows.append(rr)
            per_file_counts[fp.name][r["qa_category"]] += 1
            if copy_images and r.get("imagen"):
                cat_dir = outdir / "examples" / r["qa_category"]; copy_image_safe(r["imagen"], cat_dir)

    # global examples
    examples_csv = Path(output_dir) / "sticker_QA_examples.csv"
    with examples_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["file","pagina","qa_category","barcode_flag","hv_in_sticker","hv_outside_sticker",
                    "hv_funcion_publica_skip","image","removed_text_preview","inline_redactions"])
        for r in all_rows:
            w.writerow([r["file"], r["pagina"], r["qa_category"], r["barcode_flag"], r["hv_in_sticker"],
                        r["hv_outside_sticker"], r["hv_funcion_publica_skip"], r.get("imagen",""),
                        r.get("removed_text_preview",""), " || ".join(r.get("inline_redactions",[]))])

    summary_csv = Path(output_dir) / "sticker_QA_summary.csv"
    with summary_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); header = ["file"] + list(QA_CATEGORIES); w.writerow(header)
        for fname, cnt in per_file_counts.items():
            row = [fname] + [cnt.get(cat, 0) for cat in QA_CATEGORIES]; w.writerow(row)

    total = len(all_rows); flagged = sum(1 for r in all_rows if r["barcode_flag"])
    skipped = sum(1 for r in all_rows if r.get("hv_funcion_publica_skip"))
    print(f"[RESUMEN] archivos: {len(histories)} | páginas: {total} | con_sticker: {flagged} | HV-FP exentas: {skipped}")
    return all_rows

# ========================== EJECUCIÓN ==========================
if DO_RUN:
    _ = process_folder(JSON_FOLDER, OUTPUT_DIR, save_outputs=True, copy_images=COPY_IMAGES)


[RESUMEN] archivos: 12 | páginas: 500 | con_sticker: 238 | HV-FP exentas: 31


: 